# EDA: S&P 500 Earnings Transcripts

Goal:
- Load `kurry/sp500_earnings_transcripts` with `datasets`.
- Print basic dataset information for quick inspection.


In [1]:
from datasets import load_dataset
import pandas as pd

DATASET_ID = "kurry/sp500_earnings_transcripts"

ds = None
try:
    ds = load_dataset(DATASET_ID)
    print(f"Loaded dataset: {DATASET_ID}")
except Exception as e:
    print("Failed to load dataset. If needed, run `huggingface-cli login` and check your network.")
    print(type(e).__name__, str(e))


/Users/pandeng/miniforge3/envs/ai_narratives/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded dataset: kurry/sp500_earnings_transcripts


In [ ]:
if ds is None:
    print("Dataset is not available yet. Re-run the previous cell after network/login is fixed.")
else:
    split_names = list(ds.keys())
    print("Dataset object:", ds)
    print("Splits:", split_names)
    print()

    for split in split_names:
        subset = ds[split]
        print(f"[{split}] rows={subset.num_rows}, columns={len(subset.column_names)}")
        print("Columns:", subset.column_names)
        print("Features:", subset.features)
        print("-" * 80)


In [ ]:
if ds is None:
    print("Dataset is not available yet.")
else:
    primary_split = list(ds.keys())[0]
    subset = ds[primary_split]
    sample_size = min(5, subset.num_rows)
    sample_df = subset.select(range(sample_size)).to_pandas()

    print(f"Preview from split: {primary_split} (first {sample_size} rows)")
    display(sample_df)


In [ ]:
if ds is None:
    print("Dataset is not available yet.")
else:
    primary_split = list(ds.keys())[0]
    subset = ds[primary_split]
    inspect_rows = min(10000, subset.num_rows)
    inspect_df = subset.select(range(inspect_rows)).to_pandas()

    print(f"Missing value stats on first {inspect_rows} rows of split '{primary_split}':")
    missing = inspect_df.isna().sum().sort_values(ascending=False)
    missing_ratio = (missing / inspect_rows).round(4)
    missing_summary = pd.DataFrame({"missing_count": missing, "missing_ratio": missing_ratio})
    display(missing_summary)


In [ ]:
if ds is None:
    print("Dataset is not available yet.")
else:
    primary_split = list(ds.keys())[0]
    subset = ds[primary_split]
    inspect_rows = min(10000, subset.num_rows)
    inspect_df = subset.select(range(inspect_rows)).to_pandas()

    text_candidates = [
        c for c in inspect_df.columns
        if any(k in c.lower() for k in ["transcript", "text", "content"])
    ]

    if not text_candidates:
        text_candidates = [
            c for c in inspect_df.columns
            if inspect_df[c].dtype == "object"
        ]

    if not text_candidates:
        print("No text-like column found for length statistics.")
    else:
        text_col = text_candidates[0]
        lengths = inspect_df[text_col].fillna("").astype(str).str.len()
        print(f"Text length statistics on column '{text_col}' (first {inspect_rows} rows):")
        print(lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))


## Date-Key Split (Keep Structured Content + Company Name)

- Use `date` as the split key.
- Keep only `structured_content` and `company_name` in each split payload.


In [ ]:
if ds is None:
    print("Dataset is not available yet.")
else:
    required_cols = ["date", "symbol", "company_name", "structured_content"]
    split_df = ds["train"].to_pandas()[required_cols].copy()
    split_df["date"] = pd.to_datetime(split_df["date"], errors="coerce")

    # Only drop invalid date rows; keep rows even if symbol/company_name is missing
    split_df = split_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    # Keep only data from 2020-05-01 to 2025-05-31 (exclusive upper bound: 2025-06-01)
    start_date = pd.Timestamp("2020-05-01")
    end_date_exclusive = pd.Timestamp("2025-06-01")
    split_df = split_df[(split_df["date"] >= start_date) & (split_df["date"] < end_date_exclusive)].copy()

    # Final schema
    split_df = split_df[["date", "symbol", "company_name", "structured_content"]].copy()

    date_key_series = split_df["date"].dt.strftime("%Y-%m-%d")
    date_keyed_data = (
        split_df.groupby(date_key_series, sort=True)[["symbol", "company_name", "structured_content"]]
        .apply(lambda g: g.to_dict(orient="records"))
        .to_dict()
    )

    date_split_summary = (
        split_df.groupby(date_key_series, sort=True)
        .size()
        .rename_axis("date_key")
        .reset_index(name="row_count")
    )

    print(f"Date window: {start_date.date()} to {(end_date_exclusive - pd.Timedelta(days=1)).date()}")
    print(f"Total rows kept: {len(split_df)}")
    print(f"Total unique date keys: {len(date_keyed_data)}")
    print("Columns kept in split_df:", split_df.columns.tolist())
    print("Missing symbol rows kept:", int(split_df["symbol"].isna().sum()))
    print("Missing company_name rows kept:", int(split_df["company_name"].isna().sum()))


In [ ]:
if ds is None:
    print("Dataset is not available yet.")
else:
    print("Date split summary (head):")
    display(date_split_summary.head(10))
    print("Date split summary (tail):")
    display(date_split_summary.tail(10))

    print("Filtered dataset preview (4 columns):")
    display(split_df.head(5))

    sample_date_key = date_split_summary.iloc[-1]["date_key"]
    sample_records = date_keyed_data[sample_date_key][:2]
    print(f"Sample date key: {sample_date_key}")
    print(f"Records on that date: {len(date_keyed_data[sample_date_key])}")
    print("First 2 records (symbol/company_name/structured_content):")
    sample_records


In [ ]:
split_df.head()

## Unique Ticker Metadata (2020-05 to Latest)

Extract `ticker -> company/sector/industry` from `glopardo/sp500-earnings-transcripts` 
for records between `2020-05-01` and the latest available `earnings_date`.


In [ ]:
META_DATASET_ID = "glopardo/sp500-earnings-transcripts"
meta_ds = load_dataset(META_DATASET_ID)

meta_cols = ["ticker", "company", "sector", "industry", "earnings_date"]
meta_df = meta_ds["train"].to_pandas()[meta_cols].copy()
meta_df["earnings_date"] = pd.to_datetime(meta_df["earnings_date"], errors="coerce")

meta_df = meta_df.dropna(subset=["earnings_date", "ticker"]).copy()
start_date = pd.Timestamp("2020-05-01")
latest_date = meta_df["earnings_date"].max()

meta_window_df = meta_df[(meta_df["earnings_date"] >= start_date) & (meta_df["earnings_date"] <= latest_date)].copy()

ticker_metadata_df = (
    meta_window_df.sort_values(["ticker", "earnings_date"])
    .drop_duplicates(subset=["ticker"], keep="last")
    [["ticker", "company", "sector", "industry"]]
    .sort_values("ticker")
    .reset_index(drop=True)
)

ticker_profile_count = (
    meta_window_df[["ticker", "company", "sector", "industry"]]
    .drop_duplicates()
    .groupby("ticker")
    .size()
    .rename("profile_count")
    .reset_index()
)
ticker_profile_conflicts = ticker_profile_count[ticker_profile_count["profile_count"] > 1].copy()

print(f"Loaded metadata dataset: {META_DATASET_ID}")
print(f"Date window: {start_date.date()} to {latest_date.date()}")
print(f"Rows in date window: {len(meta_window_df)}")
print(f"Unique tickers extracted: {ticker_metadata_df['ticker'].nunique()}")
print(f"Tickers with >1 metadata profile in window: {len(ticker_profile_conflicts)}")


In [ ]:
print("Unique ticker metadata preview:")
display(ticker_metadata_df.head(20))

if len(ticker_profile_conflicts) > 0:
    print("Sample ticker profile conflicts (same ticker with multiple company/sector/industry combinations):")
    display(ticker_profile_conflicts.head(20))
else:
    print("No ticker metadata conflicts detected in this date window.")


## Merge `split_df` with `ticker_metadata_df`

- Merge on `split_df.symbol == ticker_metadata_df.ticker`
- Fill missing `company_name` from metadata `company`
- Build final ticker-keyed table with `Company_name, sector, industry, date, structured_content`


In [ ]:
required_vars = ["split_df", "ticker_metadata_df"]
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    print(f"Missing variables: {missing_vars}. Please run previous cells first.")
else:
    merge_base_df = split_df.copy()
    merge_base_df["symbol_norm"] = merge_base_df["symbol"].astype("string").str.strip().str.upper()

    meta_lookup_df = ticker_metadata_df.copy()
    meta_lookup_df["ticker"] = meta_lookup_df["ticker"].astype("string").str.strip().str.upper()

    merged_df = merge_base_df.merge(meta_lookup_df, left_on="symbol_norm", right_on="ticker", how="left")

    company_name_clean = merged_df["company_name"].replace(r"^\s*$", pd.NA, regex=True)
    merged_df["Company_name"] = company_name_clean.combine_first(merged_df["company"])

    merged_df["ticker"] = merged_df["ticker"].combine_first(merged_df["symbol_norm"])

    final_ticker_df = merged_df[["ticker", "Company_name", "sector", "industry", "date", "structured_content"]].copy()
    final_ticker_df = final_ticker_df.sort_values(["ticker", "date"]).reset_index(drop=True)

    final_ticker_table = final_ticker_df.set_index("ticker")[["Company_name", "sector", "industry", "date", "structured_content"]]

    missing_company_before = int(company_name_clean.isna().sum())
    missing_company_after = int(final_ticker_df["Company_name"].isna().sum())

    print(f"Rows in final table: {len(final_ticker_table)}")
    print(f"Unique ticker keys: {final_ticker_df['ticker'].nunique(dropna=True)}")
    print(f"Missing Company_name before fill: {missing_company_before}")
    print(f"Missing Company_name after fill: {missing_company_after}")


In [ ]:
if "final_ticker_table" not in globals():
    print("`final_ticker_table` not found. Run previous merge cell first.")
else:
    print("Final ticker-keyed table preview:")
    display(final_ticker_table.head(20))

    unresolved_company = final_ticker_df[final_ticker_df["Company_name"].isna()]
    print(f"Rows still missing Company_name: {len(unresolved_company)}")
    if len(unresolved_company) > 0:
        display(unresolved_company[["ticker", "date"]].head(20))


## Fill Remaining Missing `Company_name` via Online Lookup + Manual Overrides

Use Yahoo Finance search API for unresolved tickers, then use manual overrides for delisted/renamed symbols.


In [ ]:
import requests
from urllib.parse import quote

if "final_ticker_df" not in globals():
    print("`final_ticker_df` not found. Please run previous merge cells first.")
else:
    df_fill = final_ticker_df.copy()
    df_fill["ticker"] = df_fill["ticker"].astype("string").str.strip().str.upper()

    missing_mask_before = df_fill["Company_name"].isna() | df_fill["Company_name"].astype(str).str.strip().eq("")
    unresolved_tickers_before = sorted(df_fill.loc[missing_mask_before, "ticker"].dropna().unique().tolist())

    def yahoo_company_lookup(ticker: str, session: requests.Session) -> str | None:
        url = f"https://query1.finance.yahoo.com/v1/finance/search?q={quote(ticker)}"
        try:
            resp = session.get(url, timeout=15)
            resp.raise_for_status()
            payload = resp.json()
        except Exception:
            return None

        quotes = payload.get("quotes", []) or []

        # Strict match: exact symbol + equity type
        for q in quotes:
            symbol = str(q.get("symbol") or "").upper().strip()
            qtype = str(q.get("quoteType") or "").upper().strip()
            if symbol == ticker and qtype == "EQUITY":
                name = q.get("longname") or q.get("shortname")
                if isinstance(name, str) and name.strip():
                    return name.strip()

        return None

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    yahoo_fill_map = {}
    for t in unresolved_tickers_before:
        name = yahoo_company_lookup(t, session)
        if name:
            yahoo_fill_map[t] = name

    # Manual overrides for delisted/renamed tickers and known edge cases
    manual_company_overrides = {
        "ABMD": "Abiomed, Inc.",
        "ADS": "Alliance Data Systems Corporation",
        "ALXN": "Alexion Pharmaceuticals, Inc.",
        "ATVI": "Activision Blizzard, Inc.",
        "CERN": "Cerner Corporation",
        "CTLT": "Catalent, Inc.",
        "CXO": "Concho Resources Inc.",
        "DISCK": "Discovery, Inc.",
        "DISH": "DISH Network Corporation",
        "DRE": "Duke Realty Corporation",
        "FLIR": "FLIR Systems, Inc.",
        "FRC": "First Republic Bank",
        "HBI": "Hanesbrands Inc.",
        "HFC": "HollyFrontier Corporation",
        "INFO": "IHS Markit Ltd.",
        "JWN": "Nordstrom, Inc.",
        "KSU": "Kansas City Southern",
        "MRO": "Marathon Oil Corporation",
        "NBL": "Noble Energy, Inc.",
        "NLSN": "Nielsen Holdings plc",
        "PBCT": "People's United Financial, Inc.",
        "PXD": "Pioneer Natural Resources Company",
        "SIVB": "SVB Financial Group",
        "TWTR": "Twitter, Inc.",
        "VAR": "Varian Medical Systems, Inc.",
        "WRK": "WestRock Company",
        "XEC": "Cimarex Energy Co.",
        "XLNX": "Xilinx, Inc.",
    }

    company_name_fill_map = {}
    for t in unresolved_tickers_before:
        company_name_fill_map[t] = manual_company_overrides.get(t) or yahoo_fill_map.get(t)

    fill_mask = missing_mask_before
    candidate_fill = df_fill.loc[fill_mask, "ticker"].map(company_name_fill_map)
    df_fill.loc[fill_mask, "Company_name"] = df_fill.loc[fill_mask, "Company_name"].combine_first(candidate_fill)

    missing_mask_after = df_fill["Company_name"].isna() | df_fill["Company_name"].astype(str).str.strip().eq("")
    unresolved_tickers_after = sorted(df_fill.loc[missing_mask_after, "ticker"].dropna().unique().tolist())

    # Overwrite final artifacts for downstream reproducibility
    final_ticker_df = df_fill.copy()
    final_ticker_table = final_ticker_df.set_index("ticker")[["Company_name", "sector", "industry", "date", "structured_content"]]

    print(f"Unresolved unique tickers before online fill: {len(unresolved_tickers_before)}")
    print(f"Yahoo auto-filled tickers: {len([k for k,v in yahoo_fill_map.items() if v])}")
    print(f"Manual override candidates available: {len(manual_company_overrides)}")
    print(f"Unresolved unique tickers after fill: {len(unresolved_tickers_after)}")


In [ ]:
if "final_ticker_df" not in globals():
    print("`final_ticker_df` not found.")
else:
    missing_after = final_ticker_df[final_ticker_df["Company_name"].isna() | final_ticker_df["Company_name"].astype(str).str.strip().eq("")]
    unresolved_unique_after = sorted(missing_after["ticker"].dropna().unique().tolist())

    print(f"Remaining missing rows: {len(missing_after)}")
    print(f"Remaining missing unique tickers: {len(unresolved_unique_after)}")
    if unresolved_unique_after:
        print("Remaining unresolved tickers:")
        print(",".join(unresolved_unique_after))

    print("Updated final table preview:")
    display(final_ticker_table.head(20))


## Final Sector/Industry Standardization (Deterministic, Reproducible)

This is the only retained method for `sector` / `industry` completion and standardization.
It follows the strict two-method policy:
1. `gics_reference` direct mapping
2. `manual_mapping_table` for non-reference tickers


In [ ]:
if "final_ticker_df" not in globals():
    print("`final_ticker_df` not found. Run previous cells first.")
else:
    std_df = final_ticker_df.copy()
    std_df["ticker"] = std_df["ticker"].astype("string").str.strip().str.upper()

    gics_ref = load_dataset("jwigginton/index-constituents-sp500")["train"].to_pandas()
    gics_ref["symbol"] = gics_ref["symbol"].astype(str).str.strip().str.upper()
    gics_ref_map = (
        gics_ref[["symbol", "gics_sector", "gics_sub_industry"]]
        .drop_duplicates(subset=["symbol"], keep="last")
        .set_index("symbol")
        .to_dict(orient="index")
    )

    # Fixed base labels for non-reference tickers (manually curated table)
    manual_ticker_base_map = {
        "AAP": ("Consumer Discretionary", "Auto Parts"),
        "ABMD": ("Health Care", "Surgical & Medical Instruments"),
        "ADS": ("Financials", "Personal Credit Institutions"),
        "AIV": ("Real Estate", "REIT - Residential"),
        "ALK": ("Industrials", "Airlines"),
        "ALXN": ("Health Care", "Pharmaceutical Preparations"),
        "AMTM": ("Industrials", "Specialty Business Services"),
        "APO": ("Financials", "Asset Management & Custody Banks"),
        "ATVI": ("Communication Services", "Electronic Gaming & Multimedia"),
        "CERN": ("Information Technology", "Computer Integrated Systems Design"),
        "COTY": ("Consumer Staples", "Household & Personal Products"),
        "CPAY": ("Financials", "Transaction & Payment Processing Services"),
        "CPRI": ("Consumer Discretionary", "Luxury Goods"),
        "CRWD": ("Information Technology", "Systems Software"),
        "CXO": ("Energy", "Oil & Gas Exploration & Production"),
        "DASH": ("Consumer Discretionary", "Internet Retail"),
        "DECK": ("Consumer Discretionary", "Footwear"),
        "DELL": ("Information Technology", "Technology Hardware, Storage & Peripherals"),
        "DISCK": ("Communication Services", "Cable & Other Pay Television Services"),
        "DISH": ("Communication Services", "Telecom Services"),
        "DOC": ("Real Estate", "Health Care REITs"),
        "DRE": ("Real Estate", "Real Estate Investment Trusts"),
        "DXC": ("Technology", "Information Technology Services"),
        "ERIE": ("Financials", "Insurance Brokers"),
        "EXE": ("Energy", "Oil & Gas Exploration & Production"),
        "FBIN": ("Industrials", "Building Products & Equipment"),
        "FLIR": ("Information Technology", "Scientific & Technical Instruments"),
        "FLS": ("Industrials", "Specialty Industrial Machinery"),
        "FRC": ("Financials", "Banks - Regional"),
        "FTI": ("Energy", "Oil & Gas Equipment & Services"),
        "GDDY": ("Information Technology", "Internet Services & Infrastructure"),
        "GEV": ("Industrials", "Heavy Electrical Equipment"),
        "HBI": ("Consumer Discretionary", "Apparel Manufacturing"),
        "HFC": ("Energy", "Petroleum Refining"),
        "HOG": ("Consumer Discretionary", "Recreational Vehicles"),
        "HP": ("Energy", "Oil & Gas Drilling"),
        "HRB": ("Consumer Discretionary", "Personal Services"),
        "INFO": ("Information Technology", "Computer Programming & Data Processing"),
        "IPGP": ("Technology", "Semiconductor Equipment & Materials"),
        "JWN": ("Consumer Discretionary", "Department Stores"),
        "KKR": ("Financials", "Asset Management & Custody Banks"),
        "KSS": ("Consumer Discretionary", "Department Stores"),
        "KSU": ("Industrials", "Railroads"),
        "LEG": ("Consumer Discretionary", "Furnishings, Fixtures & Appliances"),
        "LII": ("Industrials", "Building Products"),
        "LNC": ("Financials", "Insurance - Life"),
        "LUMN": ("Communication Services", "Telecom Services"),
        "M": ("Consumer Discretionary", "Department Stores"),
        "MBC": ("Consumer Discretionary", "Furnishings, Fixtures & Appliances"),
        "NBL": ("Energy", "Oil & Gas Exploration & Production"),
        "NLSN": ("Industrials", "Business Services"),
        "NOV": ("Energy", "Oil & Gas Equipment & Services"),
        "NWL": ("Consumer Staples", "Household & Personal Products"),
        "OGN": ("Healthcare", "Drug Manufacturers - General"),
        "PBCT": ("Financials", "Banks - Regional"),
        "PENN": ("Consumer Discretionary", "Resorts & Casinos"),
        "PLTR": ("Information Technology", "Internet Services & Infrastructure"),
        "PRGO": ("Healthcare", "Drug Manufacturers - Specialty & Generic"),
        "PVH": ("Consumer Discretionary", "Apparel Manufacturing"),
        "SBNY": ("Financials", "Banks - Regional"),
        "SEDG": ("Technology", "Solar"),
        "SEE": ("Consumer Discretionary", "Packaging & Containers"),
        "SIVB": ("Financials", "Financials"),
        "SLG": ("Real Estate", "REIT - Office"),
        "SMCI": ("Information Technology", "Technology Hardware, Storage & Peripherals"),
        "SOLV": ("Health Care", "Health Care Technology"),
        "SW": ("Consumer Discretionary", "Packaging & Containers"),
        "TKO": ("Communication Services", "Entertainment"),
        "TPL": ("Energy", "Oil & Gas Exploration & Production"),
        "TWTR": ("Communication Services", "Internet Content & Information"),
        "UA": ("Consumer Discretionary", "Apparel Manufacturing"),
        "UNM": ("Financials", "Insurance - Life"),
        "VAR": ("Health Care", "Electromedical & Electrotherapeutic Apparatus"),
        "VNO": ("Real Estate", "REIT - Office"),
        "VNT": ("Technology", "Scientific & Technical Instruments"),
        "VST": ("Utilities", "Electric Utilities"),
        "WDAY": ("Information Technology", "Application Software"),
        "WSM": ("Consumer Discretionary", "Specialty Retail"),
        "WU": ("Financials", "Credit Services"),
        "XEC": ("Energy", "Oil & Gas Exploration & Production"),
        "XLNX": ("Information Technology", "Semiconductors & Related Devices"),
        "XRX": ("Industrials", "Business Equipment & Supplies"),
    }

    sector_manual_map = {
        "Technology": "Information Technology",
        "Healthcare": "Health Care",
    }

    industry_manual_map = {
        "Airlines": "Passenger Airlines",
        "Apparel Manufacturing": "Apparel, Accessories & Luxury Goods",
        "Auto Parts": "Automotive Retail",
        "Banks - Regional": "Regional Banks",
        "Building Products & Equipment": "Building Products",
        "Business Equipment & Supplies": "Technology Hardware, Storage & Peripherals",
        "Business Services": "Research & Consulting Services",
        "Cable & Other Pay Television Services": "Cable & Satellite",
        "Computer Integrated Systems Design": "IT Consulting & Other Services",
        "Computer Programming & Data Processing": "Data Processing & Outsourced Services",
        "Credit Services": "Transaction & Payment Processing Services",
        "Department Stores": "Broadline Retail",
        "Drug Manufacturers - General": "Pharmaceuticals",
        "Drug Manufacturers - Specialty & Generic": "Pharmaceuticals",
        "Electromedical & Electrotherapeutic Apparatus": "Health Care Equipment",
        "Electronic Gaming & Multimedia": "Interactive Home Entertainment",
        "Entertainment": "Movies & Entertainment",
        "Financials": "Regional Banks",
        "Footwear": "Apparel, Accessories & Luxury Goods",
        "Furnishings, Fixtures & Appliances": "Home Furnishings",
        "Health Care Technology": "Health Care Equipment",
        "Heavy Electrical Equipment": "Electrical Components & Equipment",
        "Household & Personal Products": "Household Products",
        "Information Technology Services": "IT Consulting & Other Services",
        "Insurance - Life": "Life & Health Insurance",
        "Internet Content & Information": "Interactive Media & Services",
        "Internet Retail": "Broadline Retail",
        "Luxury Goods": "Apparel, Accessories & Luxury Goods",
        "Oil & Gas Drilling": "Oil & Gas Equipment & Services",
        "Packaging & Containers": "Paper & Plastic Packaging Products & Materials",
        "Personal Credit Institutions": "Consumer Finance",
        "Personal Services": "Other Specialty Retail",
        "Petroleum Refining": "Oil & Gas Refining & Marketing",
        "Pharmaceutical Preparations": "Pharmaceuticals",
        "REIT - Office": "Office REITs",
        "REIT - Residential": "Multi-Family Residential REITs",
        "Railroads": "Rail Transportation",
        "Real Estate Investment Trusts": "Other Specialized REITs",
        "Recreational Vehicles": "Leisure Products",
        "Resorts & Casinos": "Casinos & Gaming",
        "Scientific & Technical Instruments": "Electronic Equipment & Instruments",
        "Semiconductor Equipment & Materials": "Semiconductor Materials & Equipment",
        "Semiconductors & Related Devices": "Semiconductors",
        "Solar": "Semiconductor Materials & Equipment",
        "Specialty Business Services": "Diversified Support Services",
        "Specialty Industrial Machinery": "Industrial Machinery & Supplies & Components",
        "Specialty Retail": "Other Specialty Retail",
        "Surgical & Medical Instruments": "Health Care Equipment",
        "Telecom Services": "Integrated Telecommunication Services",
    }

    sec_std = []
    ind_std = []
    src_std = []
    conf_std = []

    for _, row in std_df.iterrows():
        t = row["ticker"]
        if t in gics_ref_map:
            sec_std.append(gics_ref_map[t]["gics_sector"])
            ind_std.append(gics_ref_map[t]["gics_sub_industry"])
            src_std.append("gics_reference")
            conf_std.append("high")
        else:
            if t in manual_ticker_base_map:
                raw_sec, raw_ind = manual_ticker_base_map[t]
                src_std.append("manual_mapping_table")
                conf_std.append("high")
            else:
                raw_sec = row.get("sector")
                raw_ind = row.get("industry")
                src_std.append("manual_mapping_table")
                conf_std.append("medium")

            raw_sec = "" if pd.isna(raw_sec) else str(raw_sec).strip()
            raw_ind = "" if pd.isna(raw_ind) else str(raw_ind).strip()

            sec = sector_manual_map.get(raw_sec, raw_sec)
            ind = industry_manual_map.get(raw_ind, raw_ind)

            sec_std.append(sec if sec else pd.NA)
            ind_std.append(ind if ind else pd.NA)

    std_df["sector_std"] = sec_std
    std_df["industry_std"] = ind_std
    std_df["classification_source"] = src_std
    std_df["classification_confidence"] = conf_std

    final_ticker_std_df = std_df.copy()
    final_ticker_std_table = final_ticker_std_df.set_index("ticker")[["Company_name", "sector_std", "industry_std", "date", "structured_content", "classification_source", "classification_confidence"]]

    print("Deterministic standardization complete.")
    print(f"Rows: {len(final_ticker_std_df)}")
    print(f"Unique tickers: {final_ticker_std_df['ticker'].nunique()}")


In [ ]:
if "final_ticker_std_df" not in globals():
    print("`final_ticker_std_df` not found.")
else:
    sec_m = final_ticker_std_df["sector_std"].isna() | final_ticker_std_df["sector_std"].astype(str).str.strip().eq("")
    ind_m = final_ticker_std_df["industry_std"].isna() | final_ticker_std_df["industry_std"].astype(str).str.strip().eq("")

    gics_ref = load_dataset("jwigginton/index-constituents-sp500")["train"].to_pandas()
    valid_sec = set(gics_ref["gics_sector"].dropna().astype(str).str.strip())
    valid_ind = set(gics_ref["gics_sub_industry"].dropna().astype(str).str.strip())

    bad_sec = sorted(set(final_ticker_std_df["sector_std"].dropna().astype(str).str.strip()) - valid_sec)
    bad_ind = sorted(set(final_ticker_std_df["industry_std"].dropna().astype(str).str.strip()) - valid_ind)

    print(f"Missing sector_std rows: {int(sec_m.sum())}")
    print(f"Missing industry_std rows: {int(ind_m.sum())}")
    print(f"Missing std unique tickers: {final_ticker_std_df.loc[sec_m | ind_m, 'ticker'].nunique()}")
    print(f"Invalid sector_std labels: {len(bad_sec)}")
    if bad_sec: print(bad_sec)
    print(f"Invalid industry_std labels: {len(bad_ind)}")
    if bad_ind: print(bad_ind)

    print("classification_source distribution:")
    display(final_ticker_std_df["classification_source"].value_counts())


## Final Complete Dataset

Build a single final table and validate missing values for key columns.


In [ ]:
if "final_ticker_std_df" not in globals():
    print("`final_ticker_std_df` not found. Run previous cells first.")
else:
    final_dataset_df = final_ticker_std_df.copy()
    final_dataset_df["ticker"] = final_dataset_df["ticker"].astype("string").str.strip().str.upper()
    final_dataset_df["Company_name"] = final_dataset_df["Company_name"].replace(r"^\s*$", pd.NA, regex=True)
    final_dataset_df["date"] = pd.to_datetime(final_dataset_df["date"], errors="coerce")

    final_dataset_df = final_dataset_df.assign(
        sector=final_dataset_df["sector_std"],
        industry=final_dataset_df["industry_std"],
    )[["ticker", "Company_name", "sector", "industry", "date", "structured_content"]]
    final_dataset_df = final_dataset_df.sort_values(["date", "ticker"]).reset_index(drop=True)

    key_missing = final_dataset_df[["ticker", "sector", "industry", "date", "structured_content"]].isna().sum()
    print("Missing values in key columns:")
    display(key_missing.to_frame("missing_count"))

    print(f"Total rows: {len(final_dataset_df)}")
    print(f"Unique tickers: {final_dataset_df['ticker'].nunique(dropna=True)}")
    print(f"Date range: {final_dataset_df['date'].min()} -> {final_dataset_df['date'].max()}")
    display(final_dataset_df.head(20))
